# Data Description
- Author: 詹明翰、謝子詮、曾翊涵（共同作者；不推定個別成員已完成本單元）
- Date first created: 20261126
- Background: Unit2.1正式事故層級標準化資料；2024原51810當事人選第一當事人22368，2025為22762第一當事人，共45130候選。供Unit2.2空間歸屬及Unit2.3特徵工程，不是正式行政里A／C或最終分群樣本。
- Unit of observation: 原始第一當事人操作型候選事故紀錄；沒有官方事故唯一ID，不自動去重或當事人居民假設。
- Sample period: 2024–2025全年全天；分鐘精度、Asia/Taipei工作慣例，ISO時間含+08:00。
- Known issues: 原死亡時間窗口未完整確認，使用fatalities_source_count中性名稱；R20六／四筆保留；三筆缺座標未補值；原datum未知、4326僅工作假設。本資料沒有geometry／village_id。兩年度照明／光線不合併；年度專屬欄以_applicable辨識。速限0／502、負年齡及0／>100歲分別標記，沒有全欄刪除。候選重複僅標記，跨來源30／9筆受傷差異未覆寫。第一當事人性別／年齡／車種不代表所有涉入者或在地居民。
- Definition for each variable: 下方完整191欄字典列出dtype、單位、觀察層級、原欄映射、有效性、NA及依據；另存schema JSON，完整保留Unit1.1的49列映射及本次50個拆分來源變數。
- Data path: `data/processed/taipei_crashes_2024_2025_standardized.csv`；實際45,130列、191欄、50,400,834 bytes。
- Source data: `data/raw/113年-臺北市死傷交通事故明細.csv`（SHA `b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5`）；`data/raw/114年-臺北市死傷交通事故明細.csv`（SHA `61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c`）。八項必要證據輸入的path／hash由下方契約列示，補充原始事故ZIP／CSV不作正式觀測輸入。
- Processing code: [prepare_taipei_crashes_2024_2025.ipynb](../code/prepare_taipei_crashes_2024_2025.ipynb)，已完成乾淨kernel與獨立讀回驗證；Notebook SHA `c14dbd34bd3d776339e292b820a049b792a30ec49f2a8fc0173440ed6e146a9e`。
- Role: analysis（事故層級標準化正式資料，非訓練／測試切分或行政里最終分群資料）。
- Verification status: 本文件以獨立乾淨kernel重新讀取磁碟CSV及固定schema，實際結果見下方輸出與execution metadata；不使用處理Notebook記憶體。
- SHA-256: `b2ac0704068319b74f9aef2a28ad810d0cef473894a2a5cd23976f3fa7d2c252`。
- Downstream analysis input: `data/processed/taipei_crashes_2024_2025_standardized.csv`，與Data path完全相同。
- Schema / data dictionary: [taipei_crashes_2024_2025_standardized_schema.json](taipei_crashes_2024_2025_standardized_schema.json)，SHA `c3f064b3572d940489bdc86bb5c49dfaea8a228a3e7addbc8f9544bad324b650`。

Primary Owner：謝子詮（DE）；Collaborator / Analytical Reviewer：詹明翰（DA）；Project Reviewer：曾翊涵（PM）。本輪Codex協助執行；DA驗收、PM審查及實際個別貢獻尚待完成。此為技術交付，非人工驗收紀錄。


字典版本註：schema_mapping_49保留Unit1.1歷史提案原文；F08原暫定24h建議名已由本次中性fatalities_source_count取代。正式欄位以columns字典為準，未認證死亡時間窗，也未改寫舊Notebook。

In [1]:
from pathlib import Path
import sys,hashlib,json,csv,ast,io,platform
import pandas as pd
import numpy as np
import nbformat
from IPython.display import display,Markdown
for candidate in (Path.cwd().resolve(),*Path.cwd().resolve().parents):
    if (candidate/'AGENTS.md').is_file() and (candidate/'templates').is_dir():
        PROJECT_ROOT=candidate;break
else:raise FileNotFoundError('請從研究工作包內執行')
assert Path(sys.executable).resolve()==(PROJECT_ROOT/'.venv/Scripts/python.exe').resolve()
TEMP=PROJECT_ROOT/'data/temp/unit21_validation';TEMP.mkdir(parents=True,exist_ok=True)
def project_file(relative,area=None,must_exist=True):
    rel=Path(relative)
    assert not rel.is_absolute() and '..' not in rel.parts
    path=(PROJECT_ROOT/rel).resolve()
    allowed=PROJECT_ROOT/'data'/area if area else PROJECT_ROOT/'data'
    assert path.is_relative_to(allowed.resolve())
    if must_exist:assert path.is_file(),relative
    return path
def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()
def show(title,frame,block=18):
    display(Markdown('**'+title+'**'))
    if not len(frame):print('無符合紀錄')
    for i in range(0,len(frame),block):display(frame.iloc[i:i+block])
pd.set_option('display.max_columns',None);pd.set_option('display.max_colwidth',110)
print('Python',platform.python_version(),'pandas',pd.__version__,'numpy',np.__version__,'既有.venv／DSSI Urban Crash；不安裝套件')


Python 3.12.2 pandas 3.1.1 numpy 2.5.5 既有.venv／DSSI Urban Crash；不安裝套件


## Change for Your Own Data

固定CSV、schema及產生程式版本，任一不同即停止。UTF-8-SIG，__MISSING__為唯一衍生NA記號；來源空字串保留，不採pandas預設NA推論。類別碼／ID先字串，數值與nullable boolean再按契約明轉。日期時間存ISO字串，需日期運算時使用下方格式明轉。

In [2]:
INPUT_RELATIVE_PATH='data/processed/taipei_crashes_2024_2025_standardized.csv'
SCHEMA_RELATIVE_PATH='data/processed/taipei_crashes_2024_2025_standardized_schema.json'
EXPECTED_DATA_SHA256='b2ac0704068319b74f9aef2a28ad810d0cef473894a2a5cd23976f3fa7d2c252'
EXPECTED_SCHEMA_SHA256='c3f064b3572d940489bdc86bb5c49dfaea8a228a3e7addbc8f9544bad324b650'
EXPECTED_PRODUCER_SHA256='c14dbd34bd3d776339e292b820a049b792a30ec49f2a8fc0173440ed6e146a9e'
assert sha(project_file(INPUT_RELATIVE_PATH,'processed'))==EXPECTED_DATA_SHA256
assert sha(project_file(SCHEMA_RELATIVE_PATH,'processed'))==EXPECTED_SCHEMA_SHA256
assert sha(project_file('data/code/prepare_taipei_crashes_2024_2025.ipynb','code'))==EXPECTED_PRODUCER_SHA256
contract=json.loads(project_file(SCHEMA_RELATIVE_PATH,'processed').read_text(encoding='utf-8'))
for relative,expected in contract['source_inputs'].items():assert sha(project_file(relative))==expected,relative

In [3]:
def load_standardized(csv_path,contract):
    # All cells first read as strings: no inferred NA/ID/category conversion.
    f=pd.read_csv(csv_path,encoding=contract['csv']['encoding'],dtype='string',keep_default_na=False,na_values=[contract['csv']['missing_sentinel']])
    assert list(f)==[x['name'] for x in contract['columns']]
    for field in contract['columns']:
        name=field['name'];dtype=field['dtype'];v=f[name]
        if dtype=='boolean':
            assert v.dropna().isin(['True','False']).all(),name
            f[name]=v.map({'True':True,'False':False}).astype('boolean')
        elif dtype=='Int64':f[name]=pd.to_numeric(v,errors='raise').astype('Int64')
        elif dtype=='Float64':f[name]=v.map(lambda x:pd.NA if pd.isna(x) else float(x)).astype('Float64')
        else:assert dtype=='string'
    return f


## Summary

下列直接讀正式保存檔；來源raw只在Assertions核對追溯，不重做處理／Chapter1。

In [4]:
df=load_standardized(project_file(INPUT_RELATIVE_PATH,'processed'),contract)
print({'rows':len(df),'columns':len(df.columns),'bytes':project_file(INPUT_RELATIVE_PATH,'processed').stat().st_size,'sha256':sha(project_file(INPUT_RELATIVE_PATH,'processed')),'role':contract['role']})
show('正式資料逐年筆數',df.groupby('source_year').size().rename('N').reset_index())
show('CSV型態／缺失／時間契約',pd.DataFrame([contract['csv']]).T)
show('每項原始及規範證據輸入之固定hash',pd.DataFrame(list(contract['source_inputs'].items()),columns=['path','sha256']))

{'rows': 45130, 'columns': 191, 'bytes': 50400834, 'sha256': 'b2ac0704068319b74f9aef2a28ad810d0cef473894a2a5cd23976f3fa7d2c252', 'role': 'analysis'}


**正式資料逐年筆數**

,source_year,N
0,2024,22368
1,2025,22762


**CSV型態／缺失／時間契約**

,0
encoding,utf-8-sig
line_terminator,LF
missing_sentinel,__MISSING__
original_empty_strings,preserved; keep_default_na=False
boolean_values,"[True, False]"
date_format,%Y-%m-%d
time_format,%H:%M
datetime_format,%Y-%m-%dT%H:%M%z
timezone,Asia/Taipei; serialized +08:00


**每項原始及規範證據輸入之固定hash**

,path,sha256
0,data/raw/113年-臺北市死傷交通事故明細.csv,b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5
1,data/raw/114年-臺北市死傷交通事故明細.csv,61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c
2,data/raw/事故對照表.pdf,f40ded1ded1c816a38f421bc972cdc1320bbd280e647792207ec037d05fd3670
3,data/raw/taipei_2024_2025_crashes_raw_documentation.ipynb,0107387bd2362bd4c1a4db1056e5dbddcaad88cc362435d1236c1d7a352f4381
4,data/code/taipei_2024_2025_crash_validation.ipynb,3674d0043289f75079b46b90076795967563d721a91e303a568ecf85a25c0c6d
5,data/code/taipei_data_integration_readiness.ipynb,6048ff52d7653c9ec9d4b32cf2b78da2fb1b22dc0db0164cc48c4d2d98dd338b
6,data/code/taipei_supplementary_accident_boundary_validation.ipynb,f9df6bce7c6cdfcee95df55aa83de7187a3db3bcc76dfe76356699f6caab4dc6
7,data/code/unit17_validation_config.json,7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7


### Sample

磁碟正式資料實際前十列；raw保留字串、NA及True／False由明示契約解讀。

In [5]:
display(df.head(10))

,record_id,source_year,source_file,source_file_sha256,source_row_number,accident_year_raw,accident_month_raw,accident_day_raw,accident_hour_raw,accident_minute_raw,district_label_source_raw,district_label_source,accident_location_text_raw,accident_location_text,fatalities_source_count_raw,fatalities_source_count,fatalities_2_30d_count_raw,fatalities_2_30d_count,injured_count_raw,injured_count,party_sequence_raw,party_sequence,first_party_vehicle_code_raw,first_party_vehicle_code,first_party_vehicle_code_reference_code_known,weather_code_raw,weather_code,weather_code_reference_code_known,road_lighting_2024_code_raw,road_lighting_2024_code_applicable,road_lighting_2024_code,road_lighting_2024_code_reference_code_known,light_condition_2025_code_raw,light_condition_2025_code_applicable,light_condition_2025_code,light_condition_2025_code_reference_code_known,road_class_first_party_code_raw,road_class_first_party_code,road_class_first_party_code_reference_code_known,speed_limit_source_raw,speed_limit_source,road_shape_code_raw,road_shape_code,road_shape_code_reference_code_known,accident_position_code_raw,accident_position_code,accident_position_code_reference_code_known,surface_material_code_raw,surface_material_code,surface_material_code_reference_code_known,surface_condition_code_raw,surface_condition_code,surface_condition_code_reference_code_known,surface_defect_code_raw,surface_defect_code,surface_defect_code_reference_code_known,road_obstacle_code_raw,road_obstacle_code,road_obstacle_code_reference_code_known,sight_condition_code_raw,sight_condition_code,sight_condition_code_reference_code_known,signal_type_code_raw,signal_type_code,signal_type_code_reference_code_known,signal_status_code_raw,signal_status_code,signal_status_code_reference_code_known,direction_divider_code_raw,direction_divider_code,direction_divider_code_reference_code_known,general_lane_divider_code_raw,general_lane_divider_code,general_lane_divider_code_reference_code_known,fast_slow_lane_divider_code_raw,fast_slow_lane_divider_code,fast_slow_lane_divider_code_reference_code_known,road_edge_line_code_raw,road_edge_line_code,road_edge_line_code_reference_code_known,accident_type_code_raw,accident_type_code,accident_type_code_reference_code_known,first_party_sex_code_raw,first_party_sex_code,first_party_sex_code_reference_code_known,first_party_age_source_raw,first_party_age_source,first_party_injury_severity_code_raw,first_party_injury_severity_code,first_party_injury_severity_code_reference_code_known,first_party_injury_part_code_raw,first_party_injury_part_code,first_party_injury_part_code_reference_code_known,first_party_protection_code_raw,first_party_protection_code,first_party_protection_code_reference_code_known,first_party_phone_use_code_raw,first_party_phone_use_code,first_party_phone_use_code_reference_code_known,first_party_vehicle_use_code_raw,first_party_vehicle_use_code,first_party_vehicle_use_code_reference_code_known,first_party_action_code_raw,first_party_action_code,first_party_action_code_reference_code_known,first_party_license_status_code_raw,first_party_license_status_code,first_party_license_status_code_reference_code_known,first_party_license_type_code_raw,first_party_license_type_code,first_party_license_type_code_reference_code_known,first_party_alcohol_code_raw,first_party_alcohol_code,first_party_alcohol_code_reference_code_known,first_party_initial_impact_code_raw,first_party_initial_impact_code,first_party_initial_impact_code_reference_code_known,first_party_other_impact_code_raw,first_party_other_impact_code_applicable,first_party_other_impact_code,first_party_other_impact_code_reference_code_known,first_party_preliminary_cause_code_raw,first_party_preliminary_cause_code,first_party_preliminary_cause_code_reference_code_known,main_preliminary_cause_code_raw,main_preliminary_cause_code_applicable,main_preliminary_cause_code,main_preliminary_cause_code_reference_code_known,first_party_hit_run_code_raw,first_party_hit_run_code,fi

### Summary Stats

以下為資料說明與完整性檢查，不是正式EDA或統計推論。數值有效N及各欄缺失分開；來源空字串與序列化NA分開列示，原始適用年度也分開。

In [6]:
dictionary=pd.DataFrame(contract['columns'])
dictionary['NA_N']=[int(df[c].isna().sum()) for c in dictionary.name]
dictionary['raw_empty_string_N']=[int(df[c].eq('').sum()) if c.endswith('_raw') else 0 for c in dictionary.name]
show('完整資料字典及實際型態／缺失／原始空值',dictionary,block=12)
num=[c['name'] for c in contract['columns'] if c['dtype'] in ['Int64','Float64'] and c['name'] not in ['source_row_number','party_sequence']]
show('數值摘要；來源速限／年齡包含已明示的特殊值，另有有效欄',df[num].describe(percentiles=[.25,.5,.75]).T.reset_index())
categories=[c['name'] for c in contract['columns'] if c['name'].endswith('_code')]
for name in categories:
    v=df[name];counts=v.value_counts(dropna=False).head(10).rename_axis(name).reset_index(name='N');counts['ratio_of_all_candidates']=counts.N/len(df)
    show('類別前十項（含NA），分母45130：'+name,counts,block=10)
features=[]
for year,g in df.groupby('source_year'):
    for name in contract['features']:
        features.append(dict(year=int(year),feature=name,True_N=int(g[name].eq(True).sum()),False_N=int(g[name].eq(False).sum()),NA_N=int(g[name].isna().sum()),valid_N=int(g[name+'_valid'].sum()),candidate_N=len(g)))
show('九項三態特徵／獨立有效分母',pd.DataFrame(features))
quality_columns=['coordinate_missing_flag','speed_zero_flag','speed_502_flag','age_negative_flag','age_zero_review_flag','age_above_100_review_flag','raw_record_duplicate_flag','candidate_event_key_duplicate_flag','time_address_key_duplicate_flag','conditional_casualty_inconsistency_flag']
show('品質旗標年度True數；不同旗標可重疊，不能相加當刪除N',df.groupby('source_year')[quality_columns].sum().T)
show('R20個案原值保留',df.loc[df.conditional_casualty_inconsistency_flag.fillna(False),['record_id','source_year','first_party_injury_severity_code_raw','fatalities_source_count_raw','fatalities_2_30d_count_raw','injured_count_raw']])


**完整資料字典及實際型態／缺失／原始空值**

,name,dtype,definition,unit,source,observation_level,missing_rule,basis,NA_N,raw_empty_string_N
0,record_id,string,研究ID＝來源年＋原檔SHA256＋原CSV邏輯列；不是官方事故ID,not applicable,source_year/source_file_sha256/source_row_number,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格,0,0
1,source_year,Int64,來源檔對應西元年,year,來源版本,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格,0,0
2,source_file,string,精確原始CSV相對路徑,not applicable,固定輸入清單,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格,0,0
3,source_file_sha256,string,原始CSV完整SHA256,not applicable,固定輸入清單,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格,0,0
4,source_row_number,Int64,原CSV邏輯資料列，1起算，不含表頭；非物理行號,row,原CSV邏輯列,事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格,0,0
5,accident_year_raw,string,原字串：事故發生年，檔內為民國 113／114；對應西元時在記憶體加 1911,not applicable,"{""2024"": ""發生年度"", ""2025"": ""發生年度""}",事故時間,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 發生時間；原始值與檔名核對,0,0
6,accident_month_raw,string,原字串：事故發生月份,not applicable,"{""2024"": ""發生月"", ""2025"": ""發生月""}",事故時間,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 發生時間,0,0
7,accident_day_raw,string,原字串：事故發生日,not applicable,"{""2024"": ""發生日"", ""2025"": ""發生日""}",事故時間,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 發生時間,0,0
8,accident_hour_raw,string,原字串：事故發生小時；檔內 24 小時制數值,not applicable,"{""2024"": ""發生時-Hours"", ""2025"": ""發生時-Hours""}",事故時間,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 發生時間；檔內有效性檢查,0,0
9,accident_minute_raw,string,原字串：事故發生分鐘,not applicable,"{""2024"": ""發生分"", ""2025"": ""發生分""}",事故時間,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 發生時間,0,0


,name,dtype,definition,unit,source,observation_level,missing_rule,basis,NA_N,raw_empty_string_N
12,accident_location_text_raw,string,原字串：事故地點文字；不是當事人居住地址,not applicable,"{""2024"": ""肇事地點"", ""2025"": ""肇事地點""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 發生地點；發布 CSV 轉錄細節未知,0,0
13,accident_location_text,string,事故地點文字；不是當事人居住地址,not applicable,"{""2024"": ""肇事地點"", ""2025"": ""肇事地點""}",事故情境／總量層級（候選）,NA保留，不以0代填,PDF p.1 發生地點；發布 CSV 轉錄細節未知,0,0
14,fatalities_source_count_raw,string,原字串：原死亡人數：時間窗口尚未完整確認，中性名稱；不與2–30日死亡合併,not applicable,"{""2024"": ""死亡人數"", ""2025"": ""死亡人數""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 死傷人數；與官方頁 24 小時定義對照,0,16
15,fatalities_source_count,Int64,原死亡人數：時間窗口尚未完整確認，中性名稱；不與2–30日死亡合併,人；結構有效總量，時間／追蹤語義依欄限制,"{""2024"": ""死亡人數"", ""2025"": ""死亡人數""}",事故情境／總量層級（候選）,NA保留，不以0代填,PDF p.1 死傷人數；與官方頁 24 小時定義對照,16,0
16,fatalities_2_30d_count_raw,string,原字串：欄名表示事故 2–30 日死亡人數；彙整、追蹤及發布缺值口徑待官方確認,not applicable,"{""2024"": ""2-30日死亡人數"", ""2025"": ""2-30日死亡人數""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,CSV 欄名；PDF p.3 有個人 2–30 日死亡代碼，不能據此認證總數產製方式,0,19
17,fatalities_2_30d_count,Int64,欄名表示事故 2–30 日死亡人數；彙整、追蹤及發布缺值口徑待官方確認,人；結構有效總量，時間／追蹤語義依欄限制,"{""2024"": ""2-30日死亡人數"", ""2025"": ""2-30日死亡人數""}",事故情境／總量層級（候選）,NA保留，不以0代填,CSV 欄名；PDF p.3 有個人 2–30 日死亡代碼，不能據此認證總數產製方式,19,0
18,injured_count_raw,string,原字串：事故受傷人數；與單一當事人的受傷程度不同,not applicable,"{""2024"": ""受傷人數"", ""2025"": ""受傷人數""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 死傷人數,0,2
19,injured_count,Int64,事故受傷人數；與單一當事人的受傷程度不同,人；結構有效總量，時間／追蹤語義依欄限制,"{""2024"": ""受傷人數"", ""2025"": ""受傷人數""}",事故情境／總量層級（候選）,NA保留，不以0代填,PDF p.1 死傷人數,2,0
20,party_sequence_raw,string,原字串：事故內當事人序次的候選欄；1 為第一當事人紀錄,not applicable,"{""2024"": ""當事人序號"", ""2025"": ""當事人序號""}",第一當事人層級,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.3 第幾當事者；不是事故官方 ID，也不等同肇責認定,0,0
21,party_sequence,Int64,事故內當事人序次的候選欄；1 為第一當事人紀錄,當事人序次，非肇責,"{""2024"": ""當事人序號"", ""2025"": ""當事人序號""}",第一當事人層級,NA保留，不以0代填,PDF p.3 第幾當事者；不是事故官方 ID，也不等同肇責認定,0,0


,name,dtype,definition,unit,source,observation_level,missing_rule,basis,NA_N,raw_empty_string_N
24,first_party_vehicle_code_reference_code_known,boolean,是否在Unit1.1可核對碼集內；False含缺失／未列碼，不直接認定官方錯誤,logical flag；True/False/NA依欄定義,"{""2024"": ""車種"", ""2025"": ""車種""}",事故層級／診斷,NA保留，不以0代填,Unit1.1靜態碼表；完整性與未知見schema reference_codebooks,0,0
25,weather_code_raw,string,原字串：事故天候代碼,not applicable,"{""2024"": ""天候"", ""2025"": ""天候""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 天候；例 6 雨、7 陰、8 晴,0,2
26,weather_code,string,事故天候代碼,類別碼字串，無數值大小尺度,"{""2024"": ""天候"", ""2025"": ""天候""}",事故情境／總量層級（候選）,NA保留，不以0代填,PDF p.1 天候；例 6 雨、7 陰、8 晴,2,0
27,weather_code_reference_code_known,boolean,是否在Unit1.1可核對碼集內；False含缺失／未列碼，不直接認定官方錯誤,logical flag；True/False/NA依欄定義,"{""2024"": ""天候"", ""2025"": ""天候""}",事故層級／診斷,NA保留，不以0代填,Unit1.1靜態碼表；完整性與未知見schema reference_codebooks,0,0
28,road_lighting_2024_code_raw,string,原字串：僅2024來源道路照明設備；不認證與另一年語義等價,not applicable,"{""2024"": ""道路照明設備""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1；與 2025 光線是否同義、是否發生重新編碼尚未確認,22762,9
29,road_lighting_2024_code_applicable,boolean,來源年度是否包含此欄；False為不適用，不是來源缺失,logical flag；True/False/NA依欄定義,"{""2024"": ""道路照明設備""}",事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格,0,0
30,road_lighting_2024_code,string,僅2024來源道路照明設備；不認證與另一年語義等價,類別碼字串，無數值大小尺度,"{""2024"": ""道路照明設備""}",事故情境／總量層級（候選）,NA保留，不以0代填,PDF p.1；與 2025 光線是否同義、是否發生重新編碼尚未確認,22771,0
31,road_lighting_2024_code_reference_code_known,boolean,是否在Unit1.1可核對碼集內；False含缺失／未列碼，不直接認定官方錯誤,logical flag；True/False/NA依欄定義,"{""2024"": ""道路照明設備""}",事故層級／診斷,NA保留，不以0代填,Unit1.1靜態碼表；完整性與未知見schema reference_codebooks,0,0
32,light_condition_2025_code_raw,string,原字串：僅2025來源光線；不認證與另一年語義等價,not applicable,"{""2025"": ""光線""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1；與 2025 光線是否同義、是否發生重新編碼尚未確認,22368,12
33,light_condition_2025_code_applicable,boolean,來源年度是否包含此欄；False為不適用，不是來源缺失,logical flag；True/False/NA依欄定義,"{""2025"": ""光線""}",事故層級／診斷,NA保留，不以0代填,Unit2.1明示工作規格,0,0


,name,dtype,definition,unit,source,observation_level,missing_rule,basis,NA_N,raw_empty_string_N
36,road_class_first_party_code_raw,string,原字串：事故道路類別代碼，表單註明第 1 當事者,not applicable,"{""2024"": ""道路類別"", ""2025"": ""道路類別""}",第一當事者所在道路；PDF⑥,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1；例 5 市區道路,0,21
37,road_class_first_party_code,string,事故道路類別代碼，表單註明第 1 當事者,類別碼字串，無數值大小尺度,"{""2024"": ""道路類別"", ""2025"": ""道路類別""}",第一當事者所在道路；PDF⑥,NA保留，不以0代填,PDF p.1；例 5 市區道路,21,0
38,road_class_first_party_code_reference_code_known,boolean,是否在Unit1.1可核對碼集內；False含缺失／未列碼，不直接認定官方錯誤,logical flag；True/False/NA依欄定義,"{""2024"": ""道路類別"", ""2025"": ""道路類別""}",事故層級／診斷,NA保留，不以0代填,Unit1.1靜態碼表；完整性與未知見schema reference_codebooks,0,0
39,speed_limit_source_raw,string,原字串：道路速限，不是當事人實際車速；0、502 等值不自行更正,not applicable,"{""2024"": ""速限-速度限制"", ""2025"": ""速限-速度限制""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 道路速限；CSV 單位及 0 的發布規則待官方確認,0,5
40,speed_limit_source,Float64,道路速限，不是當事人實際車速；0、502 等值不自行更正,來源速限；km/h工作解讀，非實際速度,"{""2024"": ""速限-速度限制"", ""2025"": ""速限-速度限制""}",事故情境／總量層級（候選）,NA保留，不以0代填,PDF p.1 道路速限；CSV 單位及 0 的發布規則待官方確認,5,0
41,road_shape_code_raw,string,原字串：事故道路型態代碼,not applicable,"{""2024"": ""道路型態"", ""2025"": ""道路型態""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1；例 03 三岔、04 四岔、14 直路,0,21
42,road_shape_code,string,事故道路型態代碼,類別碼字串，無數值大小尺度,"{""2024"": ""道路型態"", ""2025"": ""道路型態""}",事故情境／總量層級（候選）,NA保留，不以0代填,PDF p.1；例 03 三岔、04 四岔、14 直路,21,0
43,road_shape_code_reference_code_known,boolean,是否在Unit1.1可核對碼集內；False含缺失／未列碼，不直接認定官方錯誤,logical flag；True/False/NA依欄定義,"{""2024"": ""道路型態"", ""2025"": ""道路型態""}",事故層級／診斷,NA保留，不以0代填,Unit1.1靜態碼表；完整性與未知見schema reference_codebooks,0,0
44,accident_position_code_raw,string,原字串：事故發生位置類別,not applicable,"{""2024"": ""事故位置"", ""2025"": ""事故位置""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1；01 路口內、02 路口附近、03 待轉區、04 停等區,0,21
45,accident_position_code,string,事故發生位置類別,類別碼字串，無數值大小尺度,"{""2024"": ""事故位置"", ""2025"": ""事故位置""}",事故情境／總量層級（候選）,NA保留，不以0代填,PDF p.1；01 路口內、02 路口附近、03 待轉區、04 停等區,21,0


,name,dtype,definition,unit,source,observation_level,missing_rule,basis,NA_N,raw_empty_string_N
48,surface_material_code,string,路面鋪裝代碼,類別碼字串，無數值大小尺度,"{""2024"": ""路面狀況1"", ""2025"": ""路面狀況1""}",事故情境／總量層級（候選）,NA保留，不以0代填,PDF p.1 路面狀況 (1),21,0
49,surface_material_code_reference_code_known,boolean,是否在Unit1.1可核對碼集內；False含缺失／未列碼，不直接認定官方錯誤,logical flag；True/False/NA依欄定義,"{""2024"": ""路面狀況1"", ""2025"": ""路面狀況1""}",事故層級／診斷,NA保留，不以0代填,Unit1.1靜態碼表；完整性與未知見schema reference_codebooks,0,0
50,surface_condition_code_raw,string,原字串：路面狀態代碼,not applicable,"{""2024"": ""路面狀況2"", ""2025"": ""路面狀況2""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 路面狀況 (2)；5 乾燥,0,21
51,surface_condition_code,string,路面狀態代碼,類別碼字串，無數值大小尺度,"{""2024"": ""路面狀況2"", ""2025"": ""路面狀況2""}",事故情境／總量層級（候選）,NA保留，不以0代填,PDF p.1 路面狀況 (2)；5 乾燥,21,0
52,surface_condition_code_reference_code_known,boolean,是否在Unit1.1可核對碼集內；False含缺失／未列碼，不直接認定官方錯誤,logical flag；True/False/NA依欄定義,"{""2024"": ""路面狀況2"", ""2025"": ""路面狀況2""}",事故層級／診斷,NA保留，不以0代填,Unit1.1靜態碼表；完整性與未知見schema reference_codebooks,0,0
53,surface_defect_code_raw,string,原字串：路面缺陷代碼,not applicable,"{""2024"": ""路面狀況3"", ""2025"": ""路面狀況3""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 路面狀況 (3)；4 無缺陷,0,21
54,surface_defect_code,string,路面缺陷代碼,類別碼字串，無數值大小尺度,"{""2024"": ""路面狀況3"", ""2025"": ""路面狀況3""}",事故情境／總量層級（候選）,NA保留，不以0代填,PDF p.1 路面狀況 (3)；4 無缺陷,21,0
55,surface_defect_code_reference_code_known,boolean,是否在Unit1.1可核對碼集內；False含缺失／未列碼，不直接認定官方錯誤,logical flag；True/False/NA依欄定義,"{""2024"": ""路面狀況3"", ""2025"": ""路面狀況3""}",事故層級／診斷,NA保留，不以0代填,Unit1.1靜態碼表；完整性與未知見schema reference_codebooks,0,0
56,road_obstacle_code_raw,string,原字串：障礙物代碼,not applicable,"{""2024"": ""道路障礙1"", ""2025"": ""道路障礙1""}",事故情境／總量層級（候選）,原始空字串不改；年度無此欄為NA；由適用旗標區分,PDF p.1 道路障礙 (1)；5 無障礙物,0,21
57,road_obstacle_code,string,障礙物代碼,類別碼字串，無數值大小尺度,"{""2024"": ""道路障礙1"", ""2025"": ""道路障礙1""}",事故情境／總量層級（候選）,NA保留，不以0代填,PDF p.1 道路障礙 (1)；5 無障礙物,21,0


... (85 outputs left out)

**品質旗標年度True數；不同旗標可重疊，不能相加當刪除N**

source_year,2024,2025
coordinate_missing_flag,1,2
speed_zero_flag,328,315
speed_502_flag,1,0
age_negative_flag,1,0
age_zero_review_flag,222,217
age_above_100_review_flag,11,18
raw_record_duplicate_flag,0,0
candidate_event_key_duplicate_flag,0,0
time_address_key_duplicate_flag,22,22
conditional_casualty_inconsistency_flag,6,4


**R20個案原值保留**

,record_id,source_year,first_party_injury_severity_code_raw,fatalities_source_count_raw,fatalities_2_30d_count_raw,injured_count_raw
2972,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r006876,2024,5,0,0,1
3986,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r009212,2024,5,0,0,1
8529,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r019829,2024,5,0,0,3
17457,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r040419,2024,5,0,0,2
17498,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r040515,2024,5,0,0,1
19983,tpe-crash:2024:b6ad23f05658531998f34ba4995a190d03872940ea48013115da89bb2a4e92c5:r046256,2024,5,0,0,1
30652,tpe-crash:2025:61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c:r008285,2025,5,0,0,2
31364,tpe-crash:2025:61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c:r008997,2025,5,0,0,1
35851,tpe-crash:2025:61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c:r013484,2025,5,0,0,2
42882,tpe-crash:2025:61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c:r020515,2025,5,0,0,2


### Assertions

核對保存檔之鍵、dtype、日期／星期、每項特徵有效性、原欄完整回查、R20及缺座標。型態與位置的合法碼集合來自固定Unit1.1證據，不用全國候選對照重新映射。

In [7]:
assert list(df)==[x['name'] for x in contract['columns']]
assert len(df)==contract['expected_rows']==45130
assert df.groupby('source_year').size().to_dict()=={2024:22368,2025:22762}
assert df.record_id.is_unique and df.record_id.notna().all() and df.party_sequence.eq(1).all()
for field in contract['columns']:assert str(df[field['name']].dtype)==field['dtype'],field['name']
assert 'geometry' not in df and 'village_id' not in df and not any('24h' in c for c in df)
dt=pd.to_datetime(df.accident_datetime,format='%Y-%m-%dT%H:%M%z',errors='raise').dt.tz_convert('Asia/Taipei')
assert dt.dt.year.eq(df.year).all() and df.year.eq(df.source_year).all()
assert dt.dt.hour.eq(df.hour).all() and dt.dt.minute.eq(df.minute).all()
assert dt.dt.dayofweek.eq(df.day_of_week).all()
assert dt.dt.strftime('%Y-%m-%d').eq(df.accident_date).all()
for feature in contract['features']:
    assert df[feature].notna().equals(df[feature+'_valid'].astype(bool))
    assert df.loc[~df[feature+'_valid'],feature].isna().all()
# Independent definition checks use hour/day and canonical string sets,
# rather than the producer's minute-of-day flag helper.
valid_type=df.accident_type_code.isin([str(k) for k in contract['reference_codebooks']['accident_type']['labels']])
valid_position=df.accident_position_code.isin([str(k) for k in contract['reference_codebooks']['position']['labels']])
valid_injury=df.injured_count.notna()&df.injured_count.ge(0)
checks={
 'morning_peak':(df.time_valid,df.hour.ge(7)&df.hour.lt(9)),
 'evening_peak':(df.time_valid,df.hour.ge(17)&df.hour.lt(19)),
 'late_night':(df.time_valid,df.hour.ge(22)|df.hour.lt(6)),
 'weekend':(df.date_valid,dt.dt.dayofweek.ge(5)),
 'intersection':(valid_position,df.accident_position_code.isin(['1','2','3','4'])),
 'intersection_strict':(valid_position,df.accident_position_code.eq('1')),
 'rear_end':(valid_type,df.accident_type_code.eq('13')),
 'side_collision':(valid_type,df.accident_type_code.eq('16')),
 'pedestrian_collision':(valid_type,df.accident_type_code.isin([str(i) for i in range(1,10)])),
 'multi_injury':(valid_injury,df.injured_count.ge(2))}
for name,(eligible,condition) in checks.items():
 expected=condition.astype('boolean').where(eligible,pd.NA)
 pd.testing.assert_series_equal(df[name],expected,check_names=False)
 pd.testing.assert_series_equal(df[name+'_valid'],eligible.astype('boolean'),check_names=False)
assert df.coordinate_missing_flag.sum()==3
assert df.groupby('source_year').conditional_casualty_inconsistency_flag.sum().to_dict()=={2024:6,2025:4}
assert df.loc[df.speed_zero_flag|df.speed_502_flag,'speed_limit_valid'].isna().all()
assert df.loc[df.age_negative_flag,'first_party_age_valid_structural'].isna().all()
assert df.groupby('source_year').accident_type_valid.sum().to_dict()=={2024:22359,2025:22750}
assert df.groupby('source_year').accident_position_valid.sum().to_dict()=={2024:22359,2025:22750}
assert df.groupby('source_year').injured_count_structural_valid.sum().to_dict()=={2024:22367,2025:22761}
assert df.groupby('source_year').fatalities_source_count_structural_valid.sum().to_dict()=={2024:22359,2025:22755}
assert df.groupby('source_year').fatalities_2_30d_count_structural_valid.sum().to_dict()=={2024:22359,2025:22752}
assert df.groupby('source_year').coordinate_parse_valid.sum().to_dict()=={2024:22367,2025:22760}
for year,g in df.groupby('source_year'):
    paths=g.source_file.unique();assert len(paths)==1
    path=str(paths[0]);assert g.source_file_sha256.eq(contract['source_inputs'][path]).all()
    expected_ids=[f'tpe-crash:{year}:{contract["source_inputs"][path]}:r{i:06d}' for i in g.source_row_number]
    assert g.record_id.tolist()==expected_ids
    raw=pd.read_csv(project_file(path,'raw'),encoding='utf-8-sig',dtype='string',keep_default_na=False)
    source=raw.iloc[g.source_row_number.astype(int).to_numpy()-1].reset_index(drop=True)
    assert pd.to_numeric(source['當事人序號']).eq(1).all()
    for spec in contract['implemented_source_variables_50']:
        name=spec['normalized_name']+'_raw';original=spec['source_columns'].get(str(year))
        if original:pd.testing.assert_series_equal(g[name].reset_index(drop=True),source[original],check_names=False)
        else:assert g[name].isna().all()
assert sha(project_file(INPUT_RELATIVE_PATH,'processed'))==EXPECTED_DATA_SHA256
assert sha(project_file(SCHEMA_RELATIVE_PATH,'processed'))==EXPECTED_SCHEMA_SHA256
for relative,expected in contract['source_inputs'].items():assert sha(project_file(relative))==expected
result=dict(status='passed',rows=len(df),columns=len(df.columns),sha256=EXPECTED_DATA_SHA256,schema_sha256=EXPECTED_SCHEMA_SHA256,independent_disk_read=True,all_source_rows_and_raw_values_checked=True,nullable_dtypes_preserved=True,spatial_assignment=False,human_acceptance='待DA／PM；本輪Codex協助')
(TEMP/'documentation_summary.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.DataFrame([result]))


C:\Users\user\OneDrive\桌面\大學\課程\院學士\資料科學與社會研究\DSSI2026-group-2\econ-5166-student-kit\data\temp\unit21_validation\kernel_temp\ipykernel_28008\527006106.py:10: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  assert dt.dt.dayofweek.eq(df.day_of_week).all()
C:\Users\user\OneDrive\桌面\大學\課程\院學士\資料科學與社會研究\DSSI2026-group-2\econ-5166-student-kit\data\temp\unit21_validation\kernel_temp\ipykernel_28008\527006106.py:24: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  'weekend':(df.date_valid,dt.dt.dayofweek.ge(5)),


,status,rows,columns,sha256,schema_sha256,independent_disk_read,all_source_rows_and_raw_values_checked,nullable_dtypes_preserved,spatial_assignment,human_acceptance
0,passed,45130,191,b2ac0704068319b74f9aef2a28ad810d0cef473894a2a5cd23976f3fa7d2c252,c3f064b3572d940489bdc86bb5c49dfaea8a228a3e7addbc8f9544bad324b650,True,True,True,False,待DA／PM；本輪Codex協助


Unit2.2以本CSV＋schema固定hash讀取，座標工作4326但原datum未知，不補三筆缺座標；正式歸里主／敏感候選需重驗。Unit2.3只對具有效歸屬的事故按各feature_valid獨立計分子分母，比例分母0保留NA。不得將本CSV冒稱里層級Dataset A／C、全部第一當事人代表全體涉入者，或設籍人口／道路密度當交通暴露。技術文件已完成不等於人工驗收；本階段結束，等待學生審查，不啟動Unit2.2。